In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

print("OpenRouter API key loaded successfully.")

In [ ]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0
)

print("Model initialized successfully")

In [ ]:
response = model.invoke(
    "Explain hypertension in simple words."
)

print(response.content)

In [ ]:
import requests
import xml.etree.ElementTree as ET

from langchain_core.tools import tool


@tool
def medical_information(topic: str) -> str:
    """
    Search MedlinePlus for general medical information about a topic.
    Provides educational information only and does not diagnose or prescribe.
    """

    url = "https://wsearch.nlm.nih.gov/ws/query"

    params = {
        "db": "healthTopics",
        "term": topic,
        "retmax": 3,
        "rettype": "brief"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        root = ET.fromstring(response.text)

        results = []

        for document in root.findall(".//document"):

            title = ""
            summary = ""
            page_url = document.attrib.get("url", "")

            for content in document.findall("content"):

                name = content.attrib.get("name")
                text = "".join(content.itertext()).strip()

                if name == "title":
                    title = text

                elif name == "full-summary":
                    summary = text

            if title or summary:
                results.append(
                    {
                        "title": title,
                        "summary": summary,
                        "url": page_url
                    }
                )

        if not results:
            return f"No medical information found for: {topic}"

        output = f"Medical information from MedlinePlus for '{topic}':\n\n"

        for i, result in enumerate(results, 1):

            output += f"{i}. {result['title']}\n"
            output += f"{result['summary']}\n"
            output += f"Source: {result['url']}\n\n"

        output += (
            "Important: This information is for educational purposes only. "
            "It does not provide a diagnosis or medical prescription."
        )

        return output

    except requests.RequestException as e:
        return f"Unable to access MedlinePlus: {str(e)}"

    except ET.ParseError:
        return "Unable to process the medical information returned by MedlinePlus."

In [ ]:
model_with_tools = model.bind_tools(
    [medical_information]
)

In [ ]:
response = model_with_tools.invoke(
    "Can you tell me about hypertension?"
)

print(response.tool_calls)

In [ ]:
! pip install -U faster-whisper sounddevice scipy

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write
from faster_whisper import WhisperModel

In [ ]:
whisper_model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("Whisper model loaded")

In [ ]:
! pip install -U edge_tts

In [ ]:
import asyncio
import edge_tts

from IPython.display import Audio, display
from langchain_core.messages import HumanMessage, ToolMessage




user_text = input("🧑 Patient: ")

print("\nPatient said:")
print(user_text)




system_prompt = """
You are a medical information voice assistant.

Your job is to provide general educational medical information.

Important rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not replace a healthcare professional.
- Use information retrieved from the medical information tool.
- Since your response will be spoken aloud, keep the answer concise.
- Use simple language.
- Do not use Markdown.
- Avoid long lists.
"""



messages = [
    HumanMessage(
        content=system_prompt
    ),
    HumanMessage(
        content=user_text
    )
]

response = model_with_tools.invoke(messages)




if response.tool_calls:

    messages.append(response)

    print("\n Tool Calling...")

    for tool_call in response.tool_calls:

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        print("Tool:", tool_name)
        print("Arguments:", tool_args)


       

        if tool_name == "medical_information":

            tool_result = medical_information.invoke(
                tool_args
            )

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )


    

    final_response = model_with_tools.invoke(
        messages
    )

    answer = final_response.content


else:

    
    # 6. IF NO TOOL IS REQUIRED
    

    answer = response.content

# 7. DISPLAY TEXT RESPONSE


print("\nMedical Assistant:")
print(answer)


async def generate_voice(text):

    voice = "en-IN-NeerjaNeural"

    communicate = edge_tts.Communicate(
        text=text,
        voice=voice
    )

    await communicate.save(
        "medical_response.mp3"
    )

await generate_voice(answer)


print("\n Medical Assistant Voice:")

display(
    Audio(
        "medical_response.mp3",
        autoplay=True
    )
)